[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/notebooks/EN/chapter4_seminar_notebook.ipynb)

# Statistics of Financial Markets — Seminar 4: Probability

*Seminar notebook. Daniel Traian Pele, Bucharest University of Economic Studies.*

- This seminar comes before Lecture 4: the slides "What You Need for Today" give every definition used here.
- Part A: computations and derivations on paper, checked in code. Part B: simulation and real data, each with an interpretation question. Part C: an open question and an AI answer to audit.
- **[Solved]**: complete code and output, a model to follow. **[Proposed]**: write your own code in the empty cell.

> The solutions are discussed in the seminar.

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_4'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

- Transparent background, legend below the plot, course palette.

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

- Daily market data from EODHD, saved in `data/market` of the SFM repository (read locally, or from GitHub in Colab); EUR/RON is the official BNR reference rate.
- Returns in %: $r_t = 100(\ln P_t - \ln P_{t-1})$; each series on its own calendar.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Seminar functions

- Daily log returns, joint returns, ACF, maximum drawdown, GBM calibration and simulation, LCG and RANDU, the binomial tree, AR(1), lag-1 dependence, conditional moments by quintile, pair statistics, yearly drawdowns.
- Helpers for Part A: discrete moments, joint tables, portfolio volatility, a two-regime mixture, binomial trees, AR(1) moments, inverse transform, GBM by hand.

In [ ]:
START = '2000-01-01'
ASSETS = ['sp500', 'dax', 'bet', 'btc']
COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F'}
SHORT = {'sp500': 'S&P 500', 'dax': 'DAX', 'bet': 'BET', 'btc': 'Bitcoin'}
PAL = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F', 'purple': '#8E44AD', 'teal': '#17A2B8', 'orange': '#E67E22', 'crimson': '#DC3545'}
RANDU_A, RANDU_M = 65539, 2147483648
SEM_START = '2000-01-01'


def returns(name, start=START, end=None):
    """Daily log returns in %, from `start` or from the first day of the series if that is later."""
    s0 = max(start, SERIES[name][4])
    return log_returns(name, s0) if end is None else log_returns(name, s0, end)


def joint_returns(a, b, start=START):
    """Daily log returns (%) of two series on their common trading days: join the prices first, then difference."""
    p = load_panel([a, b], start=start).dropna()
    return (100 * np.log(p).diff()).dropna()


def acf(x, nlags=20):
    """Sample autocorrelation function rho(1), ..., rho(nlags)."""
    x = np.asarray(x, dtype=float) - np.mean(x)
    d = np.sum(x * x)
    return np.array([np.sum(x[h:] * x[:-h]) / d for h in range(1, nlags + 1)])


def max_drawdown(prices):
    """Maximum drawdown of a price path: the largest fall from a running peak, min_t (P_t / max_{s<=t} P_s - 1)."""
    p = np.asarray(prices, dtype=float)
    return float(np.min(p / np.maximum.accumulate(p) - 1))


def gbm_params(r, ppy=None):
    """Calibrate GBM to daily log returns r (in %): annual log drift mu_log, volatility sigma, arithmetic drift
    mu = mu_log + sigma^2/2 (so that E[S_t] = S_0 exp(mu t) and median S_0 exp(mu_log t))."""
    r = pd.Series(r).dropna() / 100
    ppy = ppy or periods_per_year(r)
    m, s = r.mean() * ppy, r.std() * np.sqrt(ppy)
    return {'mu_log': m, 'sigma': s, 'mu': m + s ** 2 / 2, 'ppy': ppy, 'n': len(r)}


def simulate_gbm(S0, mu, sigma, T, steps, n_paths, seed=0):
    """Exact simulation of GBM: S_{t+dt} = S_t exp((mu - sigma^2/2) dt + sigma sqrt(dt) Z), Z ~ N(0, 1).
    Returns an array (n_paths, steps + 1). Port of SFEsimGBM."""
    rng = np.random.default_rng(seed)
    dt = T / steps
    z = rng.standard_normal((n_paths, steps))
    logp = np.cumsum((mu - sigma ** 2 / 2) * dt + sigma * np.sqrt(dt) * z, axis=1)
    return S0 * np.exp(np.hstack([np.zeros((n_paths, 1)), logp]))


def lcg(a, c, M, seed, n):
    """Linear congruential generator x_{k+1} = (a x_k + c) mod M; returns the integers and the uniforms x/M.
    Port of SFErangen1/SFErangen2."""
    x = np.empty(n, dtype=np.int64)
    x[0] = seed
    for k in range(1, n):
        x[k] = (a * int(x[k - 1]) + c) % M
    return x, x / M


def lcg_period(a, c, M, seed):
    """Length of the cycle of an LCG started at `seed`."""
    seen, x, k = {}, seed, 0
    while x not in seen:
        seen[x] = k
        x = (a * x + c) % M
        k += 1
    return k - seen[x]


def randu(n, seed=1):
    """The RANDU generator (multiplier 65539, modulus 2^31, odd seed). Port of SFErandu."""
    return lcg(RANDU_A, 0, RANDU_M, seed, n)


def binomial_crr(S0, mu, sigma, T, n):
    """Cox-Ross-Rubinstein binomial process: n steps of length dt = T/n, up factor u = exp(sigma sqrt(dt)),
    down factor d = 1/u, probability of an up move p = (exp(mu dt) - d) / (u - d), so that E[S_{t+dt}] = S_t exp(mu dt).
    Returns the terminal values S_0 u^k d^(n-k), their probabilities and the parameters. Port of SFEBinomp."""
    dt = T / n
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(mu * dt) - d) / (u - d)
    k = np.arange(n + 1)
    return {'S': S0 * u ** k * d ** (n - k), 'prob': stats.binom.pmf(k, n, p), 'u': u, 'd': d, 'p': p, 'dt': dt}


def ar1(phi, eps, x0=0.0):
    """AR(1) process X_t = phi X_{t-1} + eps_t driven by the shocks eps (phi = 1: random walk)."""
    x = np.empty(len(eps))
    prev = x0
    for t, e in enumerate(eps):
        prev = phi * prev + e
        x[t] = prev
    return x


def lag_dependence(r):
    """Corr(r_t, r_{t-1}) and Corr(r_t^2, r_{t-1}^2) with the i.i.d. band 1.96/sqrt(n)."""
    r = pd.Series(r).dropna().values
    return {'n': len(r), 'rho_r': float(np.corrcoef(r[1:], r[:-1])[0, 1]),
            'rho_r2': float(np.corrcoef(r[1:] ** 2, r[:-1] ** 2)[0, 1]),
            'rho_abs': float(np.corrcoef(np.abs(r[1:]), np.abs(r[:-1]))[0, 1]), 'band': 1.96 / np.sqrt(len(r))}


def conditional_moments(r, q=5):
    """Conditional mean and standard deviation of r_t given the quintile of |r_{t-1}|; the law of total variance
    Var(r) = E[Var(r | Q)] + Var(E[r | Q])."""
    r = pd.Series(r).dropna()
    prev = r.shift(1).abs()
    df = pd.DataFrame({'r': r, 'prev': prev}).dropna()
    df['Q'] = pd.qcut(df['prev'], q, labels=False) + 1
    g = df.groupby('Q')['r']
    mean, sd, cnt = g.mean(), g.std(ddof=0), g.count()
    w = cnt / cnt.sum()
    within = float((w * sd ** 2).sum())
    between = float((w * (mean - df['r'].mean()) ** 2).sum())
    edges = df.groupby('Q')['prev'].agg(['min', 'max'])
    return {'mean': mean.values, 'sd': sd.values, 'n': cnt.values, 'within': within, 'between': between,
            'total': float(df['r'].var(ddof=0)), 'ratio': float(sd.iloc[-1] / sd.iloc[0]),
            'sd_all': float(df['r'].std(ddof=0)), 'lo': edges['min'].values, 'hi': edges['max'].values}


def pair_stats(a, b, start=START):
    """Means, standard deviations, covariance and correlation of two return series on common days, and the
    standard deviation of the 50/50 portfolio sd_p = sqrt(w1^2 s1^2 + w2^2 s2^2 + 2 w1 w2 cov)."""
    R = joint_returns(a, b, start)
    x, y = R[a], R[b]
    cov = float(np.cov(x, y)[0, 1])
    s1, s2 = float(x.std()), float(y.std())
    sp = np.sqrt(0.25 * s1 ** 2 + 0.25 * s2 ** 2 + 0.5 * cov)
    return {'n': len(R), 'm1': float(x.mean()), 'm2': float(y.mean()), 's1': s1, 's2': s2, 'cov': cov,
            'corr': float(np.corrcoef(x, y)[0, 1]), 'sd_p': float(sp), 'sd_avg': (s1 + s2) / 2,
            'var_sum': s1 ** 2 / 4 + s2 ** 2 / 4, 'cov_term': 0.5 * cov}


def yearly_drawdowns(name):
    """Maximum drawdown within each complete calendar year (from the first close of the year)."""
    P = load_close(name, max(START, SERIES[name][4]))
    out = {}
    for y, p in P.groupby(P.index.year):
        if p.index[0].month == 1 and p.index[0].day <= 10 and p.index[-1].month == 12 and p.index[-1].day >= 15:
            out[int(y)] = max_drawdown(p.values)
    return pd.Series(out)


def discrete_moments(values, probs):
    """Discrete random variable: mean, E[X^2], variance, standard deviation and the CDF at each value."""
    x, p = np.asarray(values, float), np.asarray(probs, float)
    m = float((x * p).sum())
    m2 = float((x ** 2 * p).sum())
    return {'mean': m, 'm2': m2, 'var': m2 - m ** 2, 'sd': np.sqrt(m2 - m ** 2), 'cdf': np.cumsum(p).tolist()}


def joint_table(xs, ys, P):
    """Joint probability table P[i, j] = P(X = xs[i], Y = ys[j]): marginals, means, covariance, correlation and
    the largest gap |P(X = x, Y = y) - P(X = x) P(Y = y)| (zero for every cell iff X and Y are independent)."""
    P = np.asarray(P, float)
    x, y = np.asarray(xs, float), np.asarray(ys, float)
    px, py = P.sum(axis=1), P.sum(axis=0)
    ex, ey = (x * px).sum(), (y * py).sum()
    exy = (np.outer(x, y) * P).sum()
    vx, vy = (x ** 2 * px).sum() - ex ** 2, (y ** 2 * py).sum() - ey ** 2
    cov = exy - ex * ey
    return {'px': px.tolist(), 'py': py.tolist(), 'ex': ex, 'ey': ey, 'exy': exy, 'vx': vx, 'vy': vy, 'cov': cov,
            'corr': cov / np.sqrt(vx * vy), 'max_gap': float(np.abs(P - np.outer(px, py)).max()),
            'var_sum': vx + vy + 2 * cov}


def portfolio_sd(w, s1, s2, rho):
    """Standard deviation of w X_1 + (1 - w) X_2: sqrt(w^2 s1^2 + (1-w)^2 s2^2 + 2 w (1-w) rho s1 s2)."""
    return float(np.sqrt(w ** 2 * s1 ** 2 + (1 - w) ** 2 * s2 ** 2 + 2 * w * (1 - w) * rho * s1 * s2))


def mixture_moments(p_calm, s_calm, s_turb):
    """Two-regime model: r | calm ~ N(0, s_calm^2) with probability p_calm, r | turbulent ~ N(0, s_turb^2).
    Law of total variance Var(r) = E[Var(r | regime)] + Var(E[r | regime]) and the kurtosis E[r^4]/Var(r)^2."""
    v = p_calm * s_calm ** 2 + (1 - p_calm) * s_turb ** 2
    m4 = 3 * (p_calm * s_calm ** 4 + (1 - p_calm) * s_turb ** 4)
    return {'var': v, 'sd': np.sqrt(v), 'm4': m4, 'kurt': m4 / v ** 2, 'exkurt': m4 / v ** 2 - 3,
            'p4': p_calm * 2 * stats.norm.sf(4 * np.sqrt(v) / s_calm) + (1 - p_calm) * 2 * stats.norm.sf(4 * np.sqrt(v) / s_turb)}


def binomial_tree(S0, u, d, p, n):
    """n-step binomial tree: terminal prices S0 u^k d^(n-k), their probabilities, E[S_n], P(S_n < S0) and the
    probability p* = (1 - d)/(u - d) that makes the price a martingale."""
    k = np.arange(n + 1)
    S = S0 * u ** k * d ** (n - k)
    pr = stats.binom.pmf(k, n, p)
    return {'S': S.tolist(), 'prob': pr.tolist(), 'ES': float((S * pr).sum()), 'ES_formula': S0 * (p * u + (1 - p) * d) ** n,
            'ploss': float(pr[S < S0].sum()), 'p_star': (1 - d) / (u - d)}


def ar1_moments(c, phi, sigma, x_now=None):
    """Stationary AR(1) X_t = c + phi X_{t-1} + eps_t: mean c/(1 - phi), variance sigma^2/(1 - phi^2),
    autocorrelations phi^h and the conditional mean E[X_{t+1} | X_t = x_now] = c + phi x_now."""
    out = {'mean': c / (1 - phi), 'var': sigma ** 2 / (1 - phi ** 2), 'rho': [phi ** h for h in (1, 2, 5)]}
    if x_now is not None:
        out['cond_mean'] = c + phi * x_now
        out['cond_var'] = sigma ** 2
    return out


def inverse_transform_examples(lam=0.5, us=(0.3, 0.9), mu=0.05, sigma=1.2):
    """Inverse transform by hand: exponential F^{-1}(u) = -ln(1 - u)/lambda, a Normal quantile mu + sigma z_u."""
    return {'exp': [float(-np.log(1 - u) / lam) for u in us], 'z01': float(stats.norm.ppf(0.01)),
            'norm01': mu + sigma * stats.norm.ppf(0.01), 'norm_us': [mu + sigma * stats.norm.ppf(u) for u in us]}


def gbm_by_hand(S0, mu, sigma, T, alpha=0.05):
    """GBM: S_T = S_0 exp((mu - sigma^2/2) T + sigma W_T): mean, median, P(S_T < S_0) and the alpha quantile."""
    m, s = (mu - sigma ** 2 / 2) * T, sigma * np.sqrt(T)
    return {'mean': S0 * np.exp(mu * T), 'median': S0 * np.exp(m), 'ploss': stats.norm.cdf(-m / s),
            'q': S0 * np.exp(m + s * stats.norm.ppf(alpha)), 'm': m, 's': s, 'z': -m / s}

# Part A: computations and derivations on paper

- Do each computation on paper first; then check it with the code.

## A1 [Solved]: moments of a discrete return

**Context.** A daily return $X$ (in %) takes the values $-3$, $0$, $2$ with probabilities $0.2$, $0.3$, $0.5$.

1. Compute $E[X]$, $E[X^2]$, $\mathrm{Var}(X)$ and the standard deviation.
2. Write the CDF $F(x)$ for every $x$.
3. Compute $P(X < 0)$ and the median.

**Report:** four moments, the CDF and two numbers.

In [6]:
# Solution
pd.Series(discrete_moments([-3, 0, 2], [0.2, 0.3, 0.5]))

mean                0.4
m2                  3.8
var                3.64
sd             1.907878
cdf     [0.2, 0.5, 1.0]
dtype: object

**Interpretation of the result.** The mean is positive although a loss is possible: risk and expected return are different questions.

## A2 [Proposed]: uncorrelated but dependent

**Context.** $X \in \{-1, 0, 1\}$, $Y \in \{0, 1\}$ with $P(-1, 1) = 0.25$, $P(0, 0) = 0.5$, $P(1, 1) = 0.25$, other cells 0. Model: A1.

1. Compute the marginal distributions of $X$ and $Y$ and their means.
2. Compute $\mathrm{Cov}(X, Y)$ and $\rho$.
3. Decide whether $X$ and $Y$ are independent, using one cell of the table.
4. Compute $\mathrm{Var}(X + Y)$.

**Report:** two marginals, $\rho$, a verdict and a variance.

In [ ]:
# Your code here


## A3 [Solved]: volatility of a two-asset portfolio

**Context.** Volatilities 20% and 30%, weights 0.6 and 0.4.

1. Compute the portfolio volatility for $\rho = 0.3$.
2. Repeat for $\rho = 0$, $-0.5$ and $1$.
3. Explain when the portfolio is less risky than the weighted average of the volatilities.

**Report:** four volatilities and one sentence.

In [8]:
# Solution
pd.Series({rho: portfolio_sd(0.6, 20, 30, rho) for rho in (0.3, 0.0, -0.5, 1.0)}).round(2)

 0.3    19.35
 0.0    16.97
-0.5    12.00
 1.0    24.00
dtype: float64

**Interpretation of the result.** Only $\rho = 1$ gives the weighted average 24%; every $\rho < 1$ diversifies.

## A4 [Proposed]: two regimes and the law of total variance

**Context.** Calm with probability 0.8, $r \mid \text{calm} \sim N(0, 0.8^2)$; turbulent otherwise, $r \mid \text{turbulent} \sim N(0, 2^2)$. Model: A3.

1. Use the law of total variance to compute $\mathrm{Var}(r)$ and the standard deviation.
2. Compute $E[r^4]$ and the kurtosis $E[r^4]/\mathrm{Var}(r)^2$.
3. Explain why the mixture has heavy tails although each regime is Normal.

**Report:** a variance, a kurtosis and one sentence.

In [ ]:
# Your code here


## A5 [Solved]: a three-step binomial tree

**Context.** $S_0 = 100$, $u = 1.1$, $d = 0.9$, $p = 0.55$, three independent steps.

1. List the possible values of $S_3$ and their probabilities.
2. Compute $E[S_3]$ from the table and from $S_0(pu + (1-p)d)^3$.
3. Compute $P(S_3 < 100)$.
4. Find the $p^*$ that makes $S_t$ a martingale.

**Report:** a table, two expectations, a probability, $p^*$.

In [10]:
# Solution
binomial_tree(100, 1.1, 0.9, 0.55, 3)

{'S': [72.9, 89.10000000000002, 108.90000000000002, 133.10000000000005],
 'prob': [0.09112499999999997,
  0.33412500000000006,
  0.40837500000000004,
  0.16637500000000005],
 'ES': 103.03010000000005,
 'ES_formula': 103.03010000000002,
 'ploss': 0.42525,
 'p_star': 0.4999999999999997}

**Interpretation of the result.** With $p = 0.55 > p^* = 0.5$ the price drifts up; under $p^*$ it is a fair game.

## A6 [Proposed]: random walk, martingale and AR(1)

**Context.** $S_t = S_{t-1} + \varepsilon_t$; $X_t = 0.2 + 0.6X_{t-1} + \varepsilon_t$ with $\sigma = 1$. Model: A5.

1. Compute $E[S_t]$ and $\mathrm{Var}(S_t)$, and show that $S_t$ is a martingale.
2. Show that $S_t^2 - t\sigma^2$ is also a martingale.
3. Compute the mean, the variance and $\rho(1), \rho(2), \rho(5)$ of the stationary $X_t$.
4. Compute $E[X_{t+1} \mid X_t = 2]$ and $\mathrm{Var}(X_{t+1} \mid X_t = 2)$, and check the moments by simulation.

**Report:** two derivations, five moments, a forecast.

In [ ]:
# Your code here


## A7 [Solved]: the inverse transform step by step

**Context.** Uniform numbers $u = 0.3$, $0.9$ (and $0.15$, $0.45$, $0.8$).

1. Turn $u = 0.3$ and $u = 0.9$ into exponential draws with rate $\lambda = 0.5$.
2. Turn $u = 0.15, 0.45, 0.8$ into draws of the return $X$ of A1.
3. Turn $u = 0.01$ into a draw of $N(0.05, 1.2^2)$.

**Report:** six simulated values.

In [12]:
# Solution
print(inverse_transform_examples())
F = np.cumsum([0.2, 0.3, 0.5]); vals = np.array([-3, 0, 2])
print('A1 draws:', [vals[np.searchsorted(F, u)] for u in (0.15, 0.45, 0.8)])

{'exp': [0.7133498878774649, 4.605170185988092], 'z01': -2.3263478740408408, 'norm01': np.float64(-2.741617448849009), 'norm_us': [np.float64(-0.579280615249649), np.float64(1.5878618786535204)]}
A1 draws: [np.int64(-3), np.int64(0), np.int64(2)]


**Interpretation of the result.** A large $u$ gives a large draw: the inverse transform keeps the order of the uniforms.

## A8 [Proposed]: GBM and the Wiener process

**Context.** GBM with $\mu = 8\%$, $\sigma = 20\%$, $S_0 = 100$; $W$ a Wiener process. Model: A7.

1. Compute the mean and the median of $S_1$ and of $S_5$.
2. Compute $P(S_T < 100)$ for $T = 1$ and $T = 5$.
3. Compute the 5% quantile of $S_1$ and of $S_5$.
4. Compute $\mathrm{Var}(W(2) - W(0.5))$ and $\mathrm{Cov}(W(1), W(3))$.

**Report:** eight GBM numbers and two Wiener moments.

In [ ]:
# Your code here


# Part B: simulation, real data and interpretation

## B1 [Solved]: are S&P 500 returns independent?

**Question.** Are the daily returns of the S&P 500 (2000-2026) independent, and what does yesterday tell us about today?

1. Compute $\mathrm{Corr}(r_t, r_{t-1})$ and $\mathrm{Corr}(r_t^2, r_{t-1}^2)$ and compare them with the band $\pm 1.96/\sqrt{n}$.
2. Split the days into quintiles of $|r_{t-1}|$ and compute the mean and the standard deviation of $r_t$ in each.
3. Check the law of total variance with the quintiles as groups.
4. Interpretation: which is predictable from yesterday, the mean or the variance of today's return?

**Report:** two correlations, a table of five quintiles, one decomposition, one sentence.

In [14]:
# Solution
def b1_dependence(name='sp500', save=True):
    """B1: lag-1 correlation of returns and of squared returns, conditional mean and standard deviation of r_t by
    quintile of |r_{t-1}|, and the law of total variance, for one index."""
    r = returns(name)
    d = lag_dependence(r)
    c = conditional_moments(r)
    if save:
        fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.7))
        x, y = r.values[:-1], r.values[1:]
        axes[0].scatter(np.clip(x, -10, 10), np.clip(y, -10, 10), s=3, alpha=0.35, color=PAL['blue'],
                        label=f'(r_(t-1), r_t): correlation {d["rho_r"]:.3f}')
        axes[0].set_xlabel('r_{t-1} (%)')
        axes[0].set_ylabel('r_t (%)')
        q = np.arange(1, 6)
        axes[1].bar(q - 0.18, c['mean'], width=0.36, color=PAL['green'], label='conditional mean of r_t')
        axes[1].bar(q + 0.18, c['sd'], width=0.36, color=PAL['red'], label='conditional sd of r_t')
        axes[1].axhline(c['sd_all'], color=PAL['red'], ls='--', lw=1.1, label='unconditional sd')
        axes[1].set_xticks(q)
        axes[1].set_xticklabels(['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
        axes[1].set_xlabel('quintile of |r_{t-1}|')
        axes[1].set_ylabel('%')
        st.fig_legend_bottom(fig, ncol=2, y=0.02)
        fig.tight_layout(rect=(0, 0.14, 1, 1))
        st.check_no_grey(fig)
        st.save_fig('ch4_sem_b1')
    return {**d, 'mean': c['mean'], 'sd': c['sd'], 'ratio': c['ratio'], 'within': c['within'],
            'between': c['between'], 'total': c['total'], 'sd_all': c['sd_all']}


res = b1_dependence(save=False)
print({k: res[k] for k in ['n', 'rho_r', 'rho_r2', 'band', 'ratio', 'within', 'between', 'total']})
pd.DataFrame({'mean': res['mean'], 'sd': res['sd']}, index=['Q1', 'Q2', 'Q3', 'Q4', 'Q5']).round(3)

{'n': 6714, 'rho_r': -0.09865785288466193, 'rho_r2': 0.31332537710161545, 'band': np.float64(0.02392023284731494), 'ratio': 1.8841407765782237, 'within': 1.4689999136090206, 'between': 0.0001343467302272306, 'total': 1.4691342603392443}


,mean,sd
Q1,0.021,0.910
Q2,0.006,0.982
Q3,0.029,1.066
Q4,0.027,1.215
Q5,0.042,1.714


**Interpretation of the result.** The variance is predictable from yesterday, the mean almost not: returns are close to uncorrelated but not independent.

## B2 [Proposed]: the same for DAX, BET and Bitcoin

**Question.** Do the DAX, the BET and Bitcoin show the same pattern as the S&P 500? Model: B1.

1. Compute both lag-1 correlations and the band for each series.
2. Compute the conditional standard deviation in Q1 and Q5 and their ratio.
3. Compute the share of the variance explained by the quintile means.
4. Interpretation: why is the BET the only series with a clearly positive $\mathrm{Corr}(r_t, r_{t-1})$?

**Report:** one table and one sentence.

In [ ]:
# Your code here


## B3 [Solved]: a monthly loss probability by Monte Carlo

**Question.** How likely is a 21-day loss of more than 10% for the S&P 500, and does the answer depend on the model?

1. Compute the exact probability under i.i.d. Normal daily returns.
2. Estimate it by Monte Carlo under the same model and give the standard error.
3. Estimate it by resampling 21 observed days with replacement (the empirical inverse transform).
4. Compute the share of overlapping 21-day windows in the data with a loss beyond 10%.
5. Interpretation: which differences are simulation error and which are model error?

**Report:** four probabilities, two standard errors, one sentence.

In [16]:
# Solution
def b3_monte_carlo(name='sp500', h=21, loss=10.0, N=100000, seed=7, save=True):
    """B3: P(h-day log return < -loss %) four ways: exact under i.i.d. Normal returns, Monte Carlo under the same
    model (with its standard error), Monte Carlo by resampling the observed daily returns (bootstrap: the empirical
    inverse transform) and the empirical frequency of overlapping h-day returns."""
    r = returns(name)
    m, s = r.mean(), r.std()
    exact = stats.norm.cdf((-loss - h * m) / (s * np.sqrt(h)))
    rng = np.random.default_rng(seed)
    norm_sums = rng.normal(m, s, (N, h)).sum(axis=1)
    boot_sums = rng.choice(r.values, (N, h), replace=True).sum(axis=1)
    over = r.rolling(h).sum().dropna()
    out = {'exact': exact, 'mc_norm': float(np.mean(norm_sums < -loss)), 'mc_boot': float(np.mean(boot_sums < -loss)),
           'emp': float(np.mean(over < -loss)), 'n_over': len(over), 'N': N, 'm': m, 's': s,
           'se': float(np.sqrt(exact * (1 - exact) / N)), 'n_hits_emp': int((over < -loss).sum()),
           'emp_nonover': float(np.mean(r.values[:len(r) // h * h].reshape(-1, h).sum(axis=1) < -loss)),
           'n_nonover': len(r) // h}
    out['se_boot'] = float(np.sqrt(out['mc_boot'] * (1 - out['mc_boot']) / N))
    if save:
        fig, ax = plt.subplots(figsize=(10, 3.8))
        bins = np.linspace(-40, 25, 131)
        ax.hist(norm_sums, bins=bins, density=True, histtype='step', lw=1.6, color=PAL['red'], label='Monte Carlo, i.i.d. Normal')
        ax.hist(boot_sums, bins=bins, density=True, histtype='step', lw=1.6, color=PAL['green'], label='Monte Carlo, resampled days (bootstrap)')
        ax.hist(over, bins=bins, density=True, histtype='step', lw=1.6, color=PAL['blue'], label='observed overlapping 21-day returns')
        ax.axvline(-loss, color=st.DarkText, ls='--', lw=1.0, label='-10%')
        ax.set_yscale('log')
        ax.set_ylim(1e-5, 0.2)
        ax.set_xlabel(f'{h}-day log return (%)')
        ax.set_ylabel('density (log scale)')
        st.legend_outside_bottom(ax, ncol=2, y=-0.2)
        st.check_no_grey(fig)
        st.save_fig('ch4_sem_b3')
    return out


pd.Series(b3_monte_carlo(save=False))

exact               0.029223
mc_norm             0.029170
mc_boot             0.031840
emp                 0.026740
n_over           6694.000000
N              100000.000000
m                   0.024719
s                   1.213030
se                  0.000533
n_hits_emp        179.000000
emp_nonover         0.034483
n_nonover         319.000000
se_boot             0.000555
dtype: float64

**Interpretation of the result.** Monte Carlo and the exact value differ by less than one standard error (simulation error); the bootstrap differs from the Normal model by several standard errors (model error).

## B4 [Proposed]: test a random number generator

**Question.** Can a generator pass simple tests and still be unusable? Model: B3.

1. Compute the mean, the variance and the lag-1 correlation of RANDU and PCG64 (100,000 numbers, seed 1).
2. Count the distinct values of $9u_k - 6u_{k+1} + u_{k+2}$ for both generators.
3. Find the period of the LCGs $(5, 1, 16)$, $(5, 0, 16)$ and $(4, 1, 16)$ started at 7.
4. Interpretation: why do one-dimensional tests not detect the flaw of RANDU?

**Report:** a table of six statistics, two counts, three periods, one sentence.

In [ ]:
# Your code here


## B5 [Solved]: is the BET a geometric Brownian motion?

**Question.** Which features of the BET can a GBM with the same drift and volatility reproduce?

1. Calibrate GBM: annual log drift and volatility from the daily log returns.
2. Simulate 500 paths of the same length and compute the excess kurtosis, $\mathrm{Corr}(r_t^2, r_{t-1}^2)$ and the maximum drawdown of each.
3. Locate the real values in the simulated distributions (5% and 95% quantiles).
4. Interpretation: which stylised facts does GBM miss, and which risk number is most affected?

**Report:** two parameters, three comparisons, one sentence.

In [18]:
# Solution
def b5_gbm_check(name='bet', n_sims=500, seed=9, save=True):
    """B5: calibrate GBM to an index and compare the real series with simulations of the same length: excess
    kurtosis, Corr(r_t^2, r_{t-1}^2) and the maximum drawdown, with the rank of the real value."""
    P = load_close(name, max(START, SERIES[name][4]))
    r = 100 * np.log(P).diff().dropna()
    g = gbm_params(r)
    m, s, n = r.mean(), r.std(), len(r)
    rng = np.random.default_rng(seed)
    K, A, DD = [], [], []
    for _ in range(n_sims):
        x = rng.normal(m, s, n)
        K.append(stats.kurtosis(x))
        A.append(acf(x ** 2, 1)[0])
        DD.append(max_drawdown(np.exp(np.cumsum(x) / 100)))
    real = {'kurt': float(stats.kurtosis(r)), 'acf2': float(acf(r.values ** 2, 1)[0]), 'dd': max_drawdown(P.values)}
    out = {**g, 'n': n, 'real': real}
    for key, v in [('kurt', K), ('acf2', A), ('dd', DD)]:
        v = np.asarray(v)
        out[key] = {'q05': float(np.quantile(v, 0.05)), 'q95': float(np.quantile(v, 0.95)), 'q50': float(np.median(v)),
                    'rank': float(np.mean(v <= real[key]))}
    if save:
        fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
        for ax, (key, v, lab) in zip(axes, [('kurt', K, 'excess kurtosis'), ('acf2', A, 'Corr(r_t^2, r_{t-1}^2)'),
                                           ('dd', 100 * np.asarray(DD), 'maximum drawdown (%)')]):
            ax.hist(v, bins=40, color=PAL['teal'], alpha=0.6, label='GBM simulations' if key == 'kurt' else '_')
            ax.axvline(real[key] * (100 if key == 'dd' else 1), color=COLORS[name], lw=2,
                       label=f'real {SHORT[name]}' if key == 'kurt' else '_')
            ax.set_xlabel(lab)
        axes[0].set_ylabel('number of simulations')
        st.fig_legend_bottom(fig, ncol=2, y=0.02)
        fig.tight_layout(rect=(0, 0.1, 1, 1))
        st.check_no_grey(fig)
        st.save_fig('ch4_sem_b5')
    return out


res = b5_gbm_check(save=False)
print({k: res[k] for k in ['mu_log', 'sigma', 'n']})
pd.DataFrame({k: {'real': res['real'][k], **res[k]} for k in ['kurt', 'acf2', 'dd']}).round(3)

{'mu_log': np.float64(0.15971128299187604), 'sigma': np.float64(0.22106331764504483), 'n': 6670}


,kurt,acf2,dd
real,11.063,0.330,-0.825
q05,-0.099,-0.022,-0.628
q95,0.105,0.022,-0.314
q50,-0.008,-0.000,-0.427
rank,1.000,1.000,0.000


**Interpretation of the result.** GBM misses heavy tails and volatility clustering completely; the 2008 crash makes the real drawdown deeper than every simulated path.

## B6 [Proposed]: does variance grow linearly with the horizon?

**Question.** Does $\mathrm{Var}(h\text{-day return}) = h\,\mathrm{Var}(1\text{-day return})$ hold for the S&P 500, the BET and Bitcoin? Model: B5.

1. Compute the ratio for $h = 5, 21, 63$ with non-overlapping blocks.
2. Use the approximate standard error $\sqrt{2/(m-1)}$ with $m$ blocks to judge the distance from 1.
3. Interpretation: what does a ratio above or below 1 say about the autocorrelation of returns?

**Report:** a 3 x 3 table and two sentences.

In [ ]:
# Your code here


# Part C: open questions and AI critique

## C1 [Proposed]: how often is there a bad year?

**Question.** Does GBM give the right probability of a drawdown beyond 20% (or 40%) within one calendar year? Models: B3, B5.

1. Compute the maximum drawdown within each calendar year and the share of years beyond 20% and 40%.
2. Simulate 10,000 one-year paths under GBM and under the i.i.d. bootstrap and compute the same probabilities.
3. Compare data and models with a binomial test.
4. Propose one model that could reproduce both numbers.
5. Interpretation: what does the gap between the bootstrap and the data say about volatility clustering?

**Report:** a table, two binomial $p$-values, a short project plan.

In [ ]:
# Your code here


## C2 [Proposed]: audit an AI answer

**Context.** An AI assistant summarised the probability facts about the S&P 500 for a risk report:

- (a) the lag-1 correlation of returns is close to zero, so returns are independent and tomorrow's variance cannot be predicted;
- (b) the log price is a random walk, so its variance is the same at every date and the process is stationary;
- (c) under GBM with the S&P 500 parameters, the median value of 100 after 10 years is $100e^{10\mu}$;
- (d) a Monte Carlo probability of about 3% from 10,000 paths has a standard error of about 0.0017;
- (e) a 50/50 portfolio of S&P 500 and DAX has volatility $\sqrt{0.25\sigma_1^2 + 0.25\sigma_2^2}$.

1. For each statement, say whether it is correct; if not, give the correct statement and the correct number.

**Report:** five verdicts with one line of justification each.

In [ ]:
# Your code here
